In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Getting started.pdf', 'Colab Notebooks', 'BBC News Summary', 't5-summary-model', 'final_t5_model', 'mistral_summary_model', 'final_mistral_summary_model']


In [ ]:
base_path = "/content/drive/MyDrive/BBC News Summary/"

print(os.listdir(base_path))

['News Articles', 'Summaries']


In [ ]:

!pip install transformers datasets evaluate rouge_score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
import os
import torch
import evaluate
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [ ]:
articles = []
summaries = []

article_dir = base_path + "News Articles/business"
summary_dir = base_path + "Summaries/business"

files = sorted(os.listdir(article_dir))

for f in files[:500]:
    with open(article_dir + "/" + f, "r", encoding="latin1") as file:
        articles.append(file.read().replace("\n", " ").strip())

    with open(summary_dir + "/" + f, "r", encoding="latin1") as file:
        summaries.append(file.read().replace("\n", " ").strip())

dataset = Dataset.from_dict({
    "article": articles,
    "summary": summaries
})

print("Dataset size:", len(dataset))

Dataset size: 500


In [ ]:
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

BaseLine


In [ ]:
model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Evaluation BaseLine


In [ ]:


!pip install rouge-score nltk bert-score -q

import nltk

from rouge_score import rouge_scorer

from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction
)

from nltk.translate.meteor_score import meteor_score

from bert_score import score as bertscore


# Download nltk resources
nltk.download('wordnet')
nltk.download('omw-1.4')




baseline_predictions = []
baseline_references = []

for i in range(10):

    input_text = "summarize: " + articles[i]

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_length=60
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    baseline_predictions.append(prediction)
    baseline_references.append(summaries[i])




scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for pred, ref in zip(
    baseline_predictions,
    baseline_references
):

    scores = scorer.score(ref, pred)

    rouge1_scores.append(
        scores['rouge1'].fmeasure
    )

    rouge2_scores.append(
        scores['rouge2'].fmeasure
    )

    rougeL_scores.append(
        scores['rougeL'].fmeasure
    )

baseline_rouge1 = (
    sum(rouge1_scores) / len(rouge1_scores)
)

baseline_rouge2 = (
    sum(rouge2_scores) / len(rouge2_scores)
)

baseline_rougeL = (
    sum(rougeL_scores) / len(rougeL_scores)
)


bleu_scores = []

smooth = SmoothingFunction().method1

for pred, ref in zip(
    baseline_predictions,
    baseline_references
):

    reference = [ref.split()]
    candidate = pred.split()

    bleu = sentence_bleu(
        reference,
        candidate,
        smoothing_function=smooth
    )

    bleu_scores.append(bleu)

baseline_bleu = (
    sum(bleu_scores) / len(bleu_scores)
)


meteor_scores = []

for pred, ref in zip(
    baseline_predictions,
    baseline_references
):

    meteor = meteor_score(
        [ref.split()],
        pred.split()
    )

    meteor_scores.append(meteor)

baseline_meteor = (
    sum(meteor_scores) / len(meteor_scores)
)


P, R, F1 = bertscore(
    baseline_predictions,
    baseline_references,
    lang="en",
    verbose=True
)

baseline_bertscore = F1.mean().item()


print("\n========== BASELINE RESULTS ==========\n")

print(f"ROUGE-1  : {baseline_rouge1:.4f}")
print(f"ROUGE-2  : {baseline_rouge2:.4f}")
print(f"ROUGE-L  : {baseline_rougeL:.4f}")

print(f"BLEU     : {baseline_bleu:.4f}")

print(f"METEOR   : {baseline_meteor:.4f}")

print(f"BERTScore: {baseline_bertscore:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.74 seconds, 5.73 sentences/sec

========== BASELINE RESULTS ==========

ROUGE-1  : 0.2485
ROUGE-2  : 0.1563
ROUGE-L  : 0.1868
BLEU     : 0.0149
METEOR   : 0.1102
BERTScore: 0.8640


Pre proccessing

In [ ]:
def preprocess(examples):

    inputs = ["summarize briefly in one sentence: " + doc for doc in examples["article"]]  # better instruction

    model_inputs = tokenizer(
        inputs,
        max_length=256,   # reduce input length → less copying
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=64,    # shorter target summaries
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]  # attach labels

    return model_inputs

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/t5-summary-model",

    num_train_epochs=6,
    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    save_steps=200,
    save_total_limit=2,

    logging_steps=50
)

fine tuning

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()  # start training

Step,Training Loss
50,0.724406
100,0.719225
150,0.534167
200,0.542306
250,0.529217
300,0.494144
350,0.533874
400,0.474926
450,0.401032
500,0.498227


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.4797360610961914, metrics={'train_runtime': 227.9759, 'train_samples_per_second': 10.527, 'train_steps_per_second': 5.264, 'total_flos': 324820323532800.0, 'train_loss': 0.4797360610961914, 'epoch': 6.0})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/final_t5_model")
tokenizer.save_pretrained("/content/drive/MyDrive/final_t5_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/final_t5_model/tokenizer_config.json',
 '/content/drive/MyDrive/final_t5_model/tokenizer.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/final_t5_model")
tokenizer = T5Tokenizer.from_pretrained("/content/drive/MyDrive/final_t5_model")

Loading weights:   0%|          | 0/131 [00:01<?, ?it/s]

Fine Tuning evaluation


In [ ]:

!pip install rouge-score nltk bert-score -q

import nltk

from rouge_score import rouge_scorer

from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction
)

from nltk.translate.meteor_score import meteor_score

from bert_score import score as bertscore


# Download nltk resources
nltk.download('wordnet')
nltk.download('omw-1.4')
finetuned_predictions = []
finetuned_references = []

for i in range(10):

    input_text = "summarize: " + articles[i]

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_length=60
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    finetuned_predictions.append(prediction)

    finetuned_references.append(
        summaries[i]
    )



scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for pred, ref in zip(
    finetuned_predictions,
    finetuned_references
):

    scores = scorer.score(ref, pred)

    rouge1_scores.append(
        scores['rouge1'].fmeasure
    )

    rouge2_scores.append(
        scores['rouge2'].fmeasure
    )

    rougeL_scores.append(
        scores['rougeL'].fmeasure
    )

finetuned_rouge1 = (
    sum(rouge1_scores) / len(rouge1_scores)
)

finetuned_rouge2 = (
    sum(rouge2_scores) / len(rouge2_scores)
)

finetuned_rougeL = (
    sum(rougeL_scores) / len(rougeL_scores)
)


bleu_scores = []

smooth = SmoothingFunction().method1

for pred, ref in zip(
    finetuned_predictions,
    finetuned_references
):

    reference = [ref.split()]
    candidate = pred.split()

    bleu = sentence_bleu(
        reference,
        candidate,
        smoothing_function=smooth
    )

    bleu_scores.append(bleu)

finetuned_bleu = (
    sum(bleu_scores) / len(bleu_scores)
)

meteor_scores = []

for pred, ref in zip(
    finetuned_predictions,
    finetuned_references
):

    meteor = meteor_score(
        [ref.split()],
        pred.split()
    )

    meteor_scores.append(meteor)

finetuned_meteor = (
    sum(meteor_scores) / len(meteor_scores)
)




P, R, F1 = bertscore(
    finetuned_predictions,
    finetuned_references,
    lang="en",
    verbose=True
)

finetuned_bertscore = (
    F1.mean().item()
)


print("\n========== FINE-TUNED RESULTS ==========\n")

print(f"ROUGE-1  : {finetuned_rouge1:.4f}")

print(f"ROUGE-2  : {finetuned_rouge2:.4f}")

print(f"ROUGE-L  : {finetuned_rougeL:.4f}")

print(f"BLEU     : {finetuned_bleu:.4f}")

print(f"METEOR   : {finetuned_meteor:.4f}")

print(f"BERTScore: {finetuned_bertscore:.4f}")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.77 seconds, 12.91 sentences/sec

========== FINE-TUNED RESULTS ==========

ROUGE-1  : 0.4047
ROUGE-2  : 0.3354
ROUGE-L  : 0.3270
BLEU     : 0.1099
METEOR   : 0.2414
BERTScore: 0.8921


In [ ]:


print("\n========== IMPROVEMENT ==========\n")

print(f"ROUGE-1   : {finetuned_rouge1 - baseline_rouge1:.4f}")
print(f"ROUGE-2   : {finetuned_rouge2 - baseline_rouge2:.4f}")
print(f"ROUGE-L   : {finetuned_rougeL - baseline_rougeL:.4f}")
print(f"BLEU      : {finetuned_bleu - baseline_bleu:.4f}")
print(f"METEOR    : {finetuned_meteor - baseline_meteor:.4f}")
print(f"BERTScore : {finetuned_bertscore - baseline_bertscore:.4f}")


========== IMPROVEMENT ==========

ROUGE-1   : 0.1562
ROUGE-2   : 0.1791
ROUGE-L   : 0.1402
BLEU      : 0.0951
METEOR    : 0.1312
BERTScore : 0.0282


Baseline Summary:

time Warner profits jumped 76% to $1.13bn (£600m) for the three months to December. it said fourth quarter sales rose 2% to $11.1bn from $10.9bn. its own internet business, AOL, had mixed fortunes



Enter text (or 'exit'): TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn.For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn.Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier

Summary:
TimeWarner said fourth quarter sales rose 2% to $11 1bn from $10

Enter text (or 'exit'): exit
